In [ ]:
import os
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"  # Suppress Pygame support prompt
import pygame, sys
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import random
import tkinter as tk
from tkinter import simpledialog

from utils import (
    place_O, place_X, check_win, check_win_state, get_empty_spots,
    print_q_value, print_state_q_values, new_boards,
)
from render import (
    render_board, add_XO, check_win_update, prompt_for_vi_params,
    prompt_for_player_choice, prompt_for_eval_games, init_window,
    new_graphical_board, draw_background, draw_status_bar,
    draw_stats_panel, plot_win_rate,
)


In [ ]:
board, logical_board = new_boards()
graphical_board = new_graphical_board()

to_move = 'X'


In [ ]:
## For Value iteration we would need to generate all possible states

def generate_all_states(board, player, states):
    winner = check_win(board)
    if winner is not None:
        states.add(tuple(tuple(row) for row in board))
        return

    states.add(tuple(tuple(row) for row in board))

    for i in range(3):
        for j in range(3):
            if board[i][j] == 0:
                new_board = [row[:] for row in board]
                new_board[i][j] = player
                next_player = 1 if player == 2 else 2
                generate_all_states(new_board, next_player, states)

states = set()
initial_board = [[0, 0, 0],
                 [0, 0, 0],
                 [0, 0, 0]]
generate_all_states(initial_board, 1, states)
print(len(states))  # This will print the number of unique states generated


In [ ]:
print("Sample states:")
for i, state in enumerate(states):
    if i >= 5:  # Print only the first 5 states for brevity
        break
    print(state)
    print()


In [ ]:
## Build state values and q-values for all states and actions

state_values = {state: 0.0 for state in states}
q_values = {state: {} for state in states}

for state in states:
    for action in get_empty_spots(state):
        q_values[state][action] = 0.0

## count all state action pairs in the q_values dictionary
for state, actions in q_values.items():
    print(f"State: {state}")
    for action, value in actions.items():
        print(f"  Action: {action}, Q-value: {value}")
    print()
    break

count = 0
for state, actions in q_values.items():
    for action, value in actions.items():
        count += 1

print(f"Total state-action pairs: {count}")


In [ ]:
## Value iteration, loop through all states and actions, update state values and q-values based on expected rewards and transitions

iterations, discount = prompt_for_vi_params()

for _ in tqdm(range(iterations), desc="Value Iteration", ncols=80):  # Number of iterations
    for state in states:
        for action in get_empty_spots(state):
            # Simulate taking the action
            new_board = [list(row) for row in state]
            new_board[action[0]][action[1]] = 2  
            new_state = tuple(tuple(row) for row in new_board)

            # Check for terminal state
            winner = check_win_state(new_board)
            if winner == 1:
                reward = -1.0
            elif winner == 2:
                reward = 1.0
            elif winner == 0:
                reward = 0.0
            else:
                reward = -0.05
            
            # Update state value using Bellman equation
            max_curr_q = max(q_values[state].values(), default=0.0)
            state_values[state] = max_curr_q

            # Update Q-value using Bellman equation
            try:
                max_future_q = max(q_values[new_state].values(), default=0.0)
            except KeyError:
                q_values[new_state] = {action: 0.0 for action in get_empty_spots(new_state)}
                max_future_q = 0.0
            q_values[state][action] = reward + discount * max_future_q  


In [ ]:
## quick test

check_state = ((1, 2, 2), (1, 1, 2), (0, 1, 0))
print(max(q_values.get(check_state, {}).items(), key=lambda x: x[1]))


In [ ]:
## random play

to_move = "X"

game_finished = False

count = 0
win_count = 0
loss_count = 0
stalemate_count = 0
epsilon_greedy = 1.0

win_rate_history = []
game_intervals = []

max_episodes = prompt_for_eval_games()
player_choice = prompt_for_player_choice()

pbar = tqdm(total=max_episodes, desc="Training", ncols=80)

game_finished = True

while count < max_episodes:
    # (1) The user is about to place X if it's X's turn
    # If game is finished, do resets
    if game_finished:
        board, logical_board = new_boards()
        graphical_board = new_graphical_board()
        to_move = player_choice
        
        game_finished = False

    if to_move == "X":
        empty_spots = get_empty_spots(logical_board)
        row, col = random.choice(empty_spots)
    
        place_X(board, logical_board, (row, col))
        
        to_move = 'O'

    else:
        last_state = tuple(tuple(row) for row in logical_board)

        # pick action using the max q_value
        action = max(q_values.get(last_state, {}).items(), key=lambda x: x[1])
        row, col = action[0]

        place_O(board, logical_board, (row,col))

        new_state = tuple(tuple(row) for row in logical_board)
        to_move = 'X'
    
    winner = check_win(board)
    if winner is not None:
        if winner == "X":
            loss_count += 1
        elif winner == "O":
            win_count += 1
        else:
            stalemate_count += 1

        game_finished = True
        count += 1
        pbar.update(1) 

pbar.close()

print(f'=========== Training Results ===========')
print(f'At {count} games, the current stats are:')
print(f'Wins: {win_count}')
print(f'Losses: {loss_count}')
print(f'Stalemate: {stalemate_count}')
print(f'Current epsilon value: {epsilon_greedy}')
print(f'Win rate is {(win_count/count) * 100}%')

In [ ]:
# print_q_value(q_values)

play_count = 0
play_win_count = 0
play_loss_count = 0
play_stalemate_count = 0
epsilon_greedy = 0.5

win_rate_history = []
game_intervals = []

game_finished = True

SCREEN, BOARD, X_IMG, O_IMG, FONT, SMALL_FONT = init_window()

draw_background(SCREEN, BOARD)

pygame.display.update()

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            if play_count != 0:
                print(f'At {play_count} games, the current stats are:')
                print(f'Wins: {play_win_count}')
                print(f'Losses: {play_loss_count}')
                print(f'Stalemate: {play_stalemate_count}')
                print(f'Win rate is {(play_win_count/play_count) * 100}%')
                # print_q_value(q_values)

                plot_win_rate(game_intervals, win_rate_history)
            else:
                print("You have not played yet!")
            pygame.quit()
            sys.exit()    
        
        if event.type == pygame.MOUSEBUTTONDOWN:
            # (1) The user is about to place X if it's X's turn
            # If game is finished, do resets
            if game_finished:
                board, logical_board = new_boards()
                graphical_board = new_graphical_board()
                to_move = player_choice
                
                draw_background(SCREEN, BOARD)
                game_finished = False
                pygame.display.update()

            if to_move == "X":

                add_XO(board, graphical_board, "X", logical_board, SCREEN, X_IMG, O_IMG)

                render_board(board, X_IMG, O_IMG, graphical_board)
                for i in range(3):
                    for j in range(3):
                        if graphical_board[i][j][0] is not None:
                            SCREEN.blit(graphical_board[i][j][0], graphical_board[i][j][1])
                draw_status_bar(SCREEN, SMALL_FONT, "Agent turn")
                draw_stats_panel(SCREEN, SMALL_FONT, play_win_count, play_loss_count, play_stalemate_count, label="Value Iteration")
                pygame.display.update()

                to_move = "O"
            # The reason there has to be an else is that it should check after every move if there is a winner
            # For example, if X (you) moves last and you win, O will still go despite the game being over and then O will win 
            # Since the terminal states are checked after O in the code, even though your move (X) should've ended the game
            else:
                last_state = tuple(tuple(row) for row in logical_board)
                
                # pick action using the max q_value
                action = max(q_values.get(last_state, {}).items(), key=lambda x: x[1])
                row, col = action[0]
        
                place_O(board, logical_board, (row,col))

                render_board(board, X_IMG, O_IMG, graphical_board)
                for i in range(3):
                    for j in range(3):
                        if graphical_board[i][j][0] is not None:
                            SCREEN.blit(graphical_board[i][j][0], graphical_board[i][j][1])
                draw_status_bar(SCREEN, SMALL_FONT, "Your turn")
                draw_stats_panel(SCREEN, SMALL_FONT, play_win_count, play_loss_count, play_stalemate_count, label="Value Iteration")
                pygame.display.update()

                new_state = tuple(tuple(row) for row in logical_board)

                to_move = 'X'

            winner = check_win_update(board, graphical_board, SCREEN)
            if winner is not None:
                if winner == "X":
                    reward = -1
                    play_loss_count += 1
                elif winner == "O":
                    reward = 1
                    play_win_count += 1
                else:
                    reward = 0
                    play_stalemate_count += 1

                game_finished = True
                play_count += 1
                win_rate = play_win_count / play_count 
                win_rate_history.append(win_rate)  
                game_intervals.append(play_count) 
                if play_count % 3 == 0: 
                    print(f'At {play_count} games, the current stats are:')
                    print(f'Wins: {play_win_count}')
                    print(f'Losses: {play_loss_count}')
                    print(f'Stalemate: {play_stalemate_count}')
                    print(f'Win rate is {win_rate * 100}%')
